# Ablation SeRel-LightFM không sử dụng KGE

## Tóm tắt (Abstract)
Notebook này thực hiện thí nghiệm ablation loại bỏ hoàn toàn Knowledge Graph Embedding (KGE) khỏi SeRel-LightFM. Pipeline vẫn giữ đặc trưng metadata, đặc trưng hồ sơ người dùng và embedding văn bản được gom cụm bằng KMeans. Mục tiêu là ước lượng đóng góp riêng của thành phần KGE khi các siêu tham số LightFM và quy trình đánh giá được giữ nhất quán.

## 1. Giới thiệu (Introduction)
Ablation không KGE đóng vai trò kiểm định thực nghiệm cho giả thuyết rằng embedding đồ thị tri thức bổ sung tín hiệu quan hệ ngoài văn bản và metadata. Nếu hiệu năng suy giảm so với SeRel-LightFM đầy đủ, kết quả có thể được diễn giải như bằng chứng về vai trò của KGE; nếu không, embedding văn bản và đặc trưng cơ bản có thể đã bao phủ phần lớn thông tin cần thiết.

| Thành phần | Trạng thái trong notebook |
|---|---|
| Text Encoder | Được giữ lại |
| Item KGE | Bị loại bỏ |
| User KGE | Bị loại bỏ |
| LightFM/WARP | Giữ cùng cấu hình với mô hình đề xuất |


## 2. Phương pháp nghiên cứu (Methodology)

### 2.1 Thiết lập môi trường và cấu hình
Cell cấu hình khai báo nguồn dữ liệu, checkpoint, ngưỡng phản hồi ẩn $\tau$, siêu tham số LightFM và các tham số của từng tầng đặc trưng. Các lựa chọn này được giữ nhất quán với baseline để bảo đảm so sánh thực nghiệm công bằng.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp thư viện và khai báo toàn bộ cấu hình thực nghiệm của notebook.
# - Đầu vào chính: URL dữ liệu, thư mục checkpoint, ngưỡng phản hồi ẩn và siêu tham số mô hình.
# - Kết quả tạo ra: các hằng số cấu hình được các cell phía sau sử dụng thống nhất.
# - Lưu ý: cấu hình thuộc biến thể Ablation không KGE, do đó cần giữ cố định khi so sánh với notebook khác.

import os, pickle, ast, warnings
from collections import Counter

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Nguồn dữ liệu ──────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Cấu hình phản hồi ẩn ─────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7
MIN_RATINGS        = 10

# ── Siêu tham số LightFM (giống KGE-UserNode để so sánh công bằng) ───────────
NO_COMPONENTS  = 64
LEARNING_RATE  = 0.05
ITEM_ALPHA     = 1e-6
USER_ALPHA     = 1e-6
MAX_EPOCHS     = 30
PATIENCE       = 5
NUM_THREADS    = 4
K_VALUES       = (5, 10, 20, 50)

# ── Tầng 2: mã hóa văn bản ────────────────────────────────────────────────────
TEXT_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
K_TEXT_CLUSTERS = 200
TEXT_TOP_K      = 3
TEXT_TEMP       = 0.1

# ── Cờ điều khiển ablation ────────────────────────────────────────────────────────────
USE_TEXT_CLUSTERS    = True
USE_KG_CLUSTERS      = False  # [ABLATION] KGE item clusters DISABLED
USE_USER_KG_CLUSTERS = False  # [ABLATION] KGE user clusters DISABLED

# ── GPU Cấu hình thực nghiệm ─────────────────────────────────────────────────────────
import torch

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_STR = "cuda" if torch.cuda.is_available() else "cpu"

TEXT_BATCH_SIZE = 512 if DEVICE.type == "cuda" else 64

print(f"[Ablation-NoKGE] Cấu hình: threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}, "
      f"lr={LEARNING_RATE}, patience={PATIENCE}")
print(f"[Ablation-NoKGE] Split : Leave-One-Out (LOO) per user")
print(f"[Ablation-NoKGE] Tier 2 Text   : {TEXT_MODEL_NAME} → KMeans({K_TEXT_CLUSTERS}) → soft top-{TEXT_TOP_K}")
print(f"[Ablation-NoKGE] Tier 2 KGE    : DISABLED (ablation)")
print(f"[Ablation-NoKGE] Tier 3 User   : binary flags + hour/acc_year/weekday")
print()
print(f"[GPU] Device: {DEVICE_STR.upper()}")
if DEVICE.type == "cuda":
    print(f"[GPU] Name  : {torch.cuda.get_device_properties(0).name}")
else:
    print("[GPU] ⚠ CUDA không khả dụng — chạy trên CPU")


### 2.2 Các hàm hỗ trợ
Nhóm hàm hỗ trợ thực hiện chuẩn hóa văn bản, token hóa, lọc token hiếm và xử lý giá trị thiếu. Đây là phần tiền xử lý dùng chung cho toàn bộ pipeline.


In [ ]:
# [Giải thích cell]
# - Mục đích: định nghĩa các hàm tiền xử lý văn bản và token dùng lại trong notebook.
# - Đầu vào chính: các cột metadata dạng chuỗi, thường chứa token phân tách bằng dấu `|`.
# - Kết quả tạo ra: chuỗi đã làm sạch, danh sách token và tập token đủ phổ biến để làm đặc trưng.
# - Lưu ý: các hàm này chỉ chuẩn hóa dữ liệu; không học tham số từ validation hoặc test.

def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col


def tokenize(series, sep='|'):
    """Split pipe-separated Series → list-of-lists. NaN → []."""
    return [
        [t.strip() for t in s.split(sep) if t.strip()]
        if isinstance(s, str) else []
        for s in series
    ]


def filter_rare(token_lists, min_count):
    """Loại token xuất hiện < min_count lần."""
    if min_count <= 1:
        return token_lists
    counter = Counter()
    for toks in token_lists:
        counter.update(set(toks))
    keep = {t for t, c in counter.items() if c >= min_count}
    return [[t for t in toks if t in keep] for toks in token_lists]


def fix_plot(p):
    """Fix byte-string encoded plots và encoding artifacts."""
    if not isinstance(p, str) or p.strip() in ('', 'nan'):
        return ""
    if p.startswith("b'") or p.startswith('b"'):
        try:
            decoded = ast.literal_eval(p)
            if isinstance(decoded, bytes):
                return decoded.decode('utf-8', errors='replace')
        except Exception:
            try:
                decoded = ast.literal_eval(p)
                if isinstance(decoded, bytes):
                    return decoded.decode('latin-1', errors='replace')
            except Exception:
                return p[2:-1]
    return p


print("Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot")

Helpers defined: cleanup_column, tokenize, filter_rare, fix_plot


### 2.3 Nạp dữ liệu
Các bảng `movies.csv`, `ratings.csv` và `user_profiles.csv` được nạp và chuẩn hóa kiểu thời gian. Dữ liệu rating cung cấp tín hiệu tương tác, trong khi metadata phim và hồ sơ người dùng cung cấp đặc trưng phụ.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp ba bảng dữ liệu gốc gồm rating, hồ sơ người dùng và metadata phim.
# - Đầu vào chính: các đường dẫn `RATINGS_URL`, `USERS_URL` và `MOVIES_URL`.
# - Kết quả tạo ra: `ratings_df`, `users_df`, `movies_df` và cột thời gian đã được chuẩn hóa.
# - Lưu ý: kiểm tra kích thước bảng sau khi nạp giúp phát hiện sớm lỗi đọc dữ liệu.

ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


### 2.4 Lọc dữ liệu hợp lệ
Các rating không có hồ sơ người dùng, người dùng có quá ít rating và movie không xuất hiện trong metadata được loại bỏ. Bước này bảo đảm tập dữ liệu còn lại phù hợp với Leave-One-Out và xây dựng đặc trưng.


In [ ]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

# 2.1 Lọc orphan users
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


### 2.5 Chuẩn hóa văn bản metadata
Các trường văn bản phân tách bằng dấu `|` và trường `plot` được làm sạch để giảm lỗi encoding. Bước này tạo đầu vào ổn định cho token hóa và mã hóa văn bản.


In [ ]:
# [Giải thích cell]
# - Mục đích: làm sạch lỗi encoding trong metadata phim trước khi token hóa hoặc hiển thị.
# - Đầu vào chính: các cột văn bản như `genres`, `topics`, `actors`, `directors` và `plot`.
# - Kết quả tạo ra: metadata nhất quán hơn, giảm nhiễu cho đặc trưng nội dung và text encoder.
# - Lưu ý: đây là xử lý thuộc tính item, không dùng nhãn validation/test để học mô hình.

# 3.1 Fix encoding artifacts
for col in ["country_name", "genres", "topics", "directors", "actors",
            "script", "producer", "music", "photography"]:
    movies_df[col] = cleanup_column(movies_df[col])

# 3.2 Fix byte-string plots
movies_df["plot_clean"] = movies_df["plot"].apply(fix_plot)

print("Text cleanup hoàn tất.")
print(f"Plot NaN/empty: {(movies_df['plot_clean'] == '').sum():,} / {len(movies_df):,}")

Text cleanup hoàn tất.
Plot NaN/empty: 2,442 / 78,628


### 2.6 Chia dữ liệu Leave-One-Out
Với mỗi người dùng, tương tác mới nhất được dùng làm test, tương tác liền trước dùng làm validation và phần còn lại dùng làm train. Mọi quan hệ user-item trong đồ thị tri thức chỉ được xây dựng từ train để tránh rò rỉ dữ liệu.


In [ ]:
# [Giải thích cell]
# - Mục đích: chia dữ liệu theo Leave-One-Out ở cấp người dùng.
# - Cách làm: sắp xếp rating theo thời gian, lấy tương tác mới nhất làm test và tương tác liền trước làm validation.
# - Kết quả tạo ra: `train_df`, `valid_df`, `test_df` và các tập held-out dương sau khi áp ngưỡng.
# - Lưu ý: chỉ phần train được dùng để xây dựng mô hình và các quan hệ user-item.

ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx, valid_idx, test_idx = [], [], []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")

Train : 962,280 ratings  |  474 users
Valid :     474 ratings  |  474 users  (1 rating/user)
Test  :     474 ratings  |  474 users  (1 rating/user)


### 2.7 Phân loại item warm/cold
Item warm đã xuất hiện trong train, còn item cold chưa xuất hiện trong train. Phân tách này giúp đánh giá riêng khả năng tổng quát hóa của mô hình đối với item thiếu lịch sử tương tác.


In [ ]:
# [Giải thích cell]
# - Mục đích: tách item trong tập kiểm thử thành warm và cold theo sự xuất hiện trong train.
# - Đầu vào chính: `train_df` và `test_df`.
# - Kết quả tạo ra: `test_warm_df`, `test_cold_df` cùng các phiên bản positive nếu có.
# - Lưu ý: phân tách này giúp đánh giá riêng khả năng xử lý item cold-start.

movies_in_train = set(train_df["movie_id"].unique())

test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"\nTEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)")

Unique items in train : 75,115

TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)
TEST cold  :     28 ratings  (5.9%)


### 2.8 Xây dựng Tier 1 item features
Tier 1 gồm các đặc trưng nội dung có cấu trúc như `genre`, `topic`, `country`, `year` và `duration`. Biến liên tục được chuẩn hóa trước khi đưa vào từ điển đặc trưng item.


In [ ]:
# [Giải thích cell]
# - Mục đích: xây dựng đặc trưng item có cấu trúc từ metadata phim.
# - Đầu vào chính: thể loại, chủ đề, quốc gia, năm phát hành và thời lượng.
# - Kết quả tạo ra: `item_feat_dicts`, trong đó mỗi movie_id ánh xạ tới các feature có trọng số.
# - Lưu ý: biến liên tục được chuẩn hóa để tránh lấn át đặc trưng phân loại.

# Token hóa các cột phân loại
genre_toks   = tokenize(movies_df["genres"])
topic_toks   = tokenize(movies_df["topics"])
country_toks = filter_rare(tokenize(movies_df["country_name"]), min_count=2)

# Tham số chuẩn hóa biến liên tục
year_min   = movies_df["year_published"].min()
year_range = movies_df["year_published"].max() - year_min
dur_cap    = movies_df["duration"].quantile(0.99)

# Xây dựng từ điển đặc trưng item: {movie_id: {feature_name: weight}}
item_feat_dicts = {}
for i in range(len(movies_df)):
    row = movies_df.iloc[i]
    mid = int(row["id"])
    feats = {}

    if pd.notna(row["year_published"]) and year_range > 0:
        feats["year"] = (row["year_published"] - year_min) / year_range
    if pd.notna(row["duration"]) and dur_cap > 0:
        feats["duration"] = min(row["duration"] / dur_cap, 1.0)

    for g in genre_toks[i]:
        feats[f"genre:{g}"] = 1.0
    for t in topic_toks[i]:
        feats[f"topic:{t}"] = 1.0
    for c in country_toks[i]:
        feats[f"country:{c}"] = 1.0

    item_feat_dicts[mid] = feats

# Tóm tắt kiểm tra
all_tier1_tags = set()
for feats in item_feat_dicts.values():
    all_tier1_tags.update(feats.keys())

print(f"Tier 1 features: {len(all_tier1_tags)} unique tags")
print(f"  genre/topic/country/continuous → giữ nguyên 100% baseline")

Tier 1 features: 590 unique tags
  genre/topic/country/continuous → giữ nguyên 100% baseline


### 2.9 Xây dựng Tier 3 user features
Tier 3 gồm các đặc trưng hồ sơ tĩnh của người dùng. Việc giữ lại nhóm đặc trưng này giúp LightFM khai thác side information ở phía user ngoài tín hiệu tương tác.


In [ ]:
# [Giải thích cell]
# - Mục đích: chuyển hồ sơ người dùng thành đặc trưng rời rạc cho LightFM.
# - Đầu vào chính: các cột hành vi và hồ sơ tĩnh trong `users_df`.
# - Kết quả tạo ra: `user_feat_dicts`, ánh xạ user_id tới tập feature mô tả người dùng.
# - Lưu ý: đặc trưng user không lấy từ validation/test, giúp giảm nguy cơ rò rỉ dữ liệu.

def bin_preferred_hour(h):
    if pd.isna(h):       return "hour:unknown"
    if 5 <= h <= 11:     return "hour:morning"
    elif 12 <= h <= 17:  return "hour:afternoon"
    elif 18 <= h <= 22:  return "hour:evening"
    else:                return "hour:night"

def bin_account_year(y):
    if pd.isna(y):     return "acc:unknown"
    if y < 2010:       return "acc:pre2010"
    elif y <= 2013:    return "acc:2010_2013"
    else:              return "acc:post2013"

users_df["hour_bin"]     = users_df["preferred_hour"].apply(bin_preferred_hour)
users_df["acc_year_bin"] = users_df["account_creation_year"].apply(bin_account_year)
users_df["weekday_bin"]  = "weekday:" + users_df["preferred_weekday"].astype(str)

BINARY_FLAGS = ["night_owl", "early_bird", "weekend_tweeter", "week_tweeter", "geo_enabled"]

user_feat_dicts = {}
for _, row in users_df.iterrows():
    uid   = int(row["id"])
    feats = {}

    for col in BINARY_FLAGS:
        if row.get(col, 0) == 1:
            feats[f"user:{col}"] = 1.0

    feats[row["hour_bin"]]     = 1.0
    feats[row["acc_year_bin"]] = 1.0
    feats[row["weekday_bin"]]  = 1.0

    user_feat_dicts[uid] = feats

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

print(f"Tier 3 user features: {len(all_user_tags)} unique tags")
print(f"  Users với features: {len(user_feat_dicts)} / {users_df['id'].nunique()}")

Tier 3 user features: 19 unique tags
  Users với features: 474 / 474


### 2.10 Mã hóa văn bản bằng Sentence Transformer
Các trường văn bản của item được ghép thành đầu vào đa trường và mã hóa thành embedding ngữ nghĩa. Trong ablation này, embedding văn bản là nguồn Tier 2 duy nhất được giữ lại.


In [ ]:
# [Giải thích cell]
# - Mục đích: cài các thư viện cần thiết để mã hóa văn bản và gom cụm embedding.
# - Lưu ý: cell này chỉ chuẩn bị môi trường; không tạo dữ liệu huấn luyện hay kết quả đánh giá.

pip install sentence-transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# [Giải thích cell]
# - Mục đích: tạo chuỗi mô tả đa trường cho từng phim và mã hóa thành embedding văn bản.
# - Đầu vào chính: title, genres, topics, country và plot đã được làm sạch.
# - Kết quả tạo ra: `text_embeddings`, ma trận biểu diễn ngữ nghĩa của item.
# - Lưu ý: embedding được chuẩn hóa L2 để cosine similarity có thể tính bằng tích vô hướng.

from sentence_transformers import SentenceTransformer

# ── Bước 1: xây dựng đầu vào văn bản đa trường và bỏ qua trường rỗng ──
def build_text_input(row):
    parts = []
    title = row.get("original_title")
    if isinstance(title, str) and title.strip():
        parts.append(title.strip())

    genres = row.get("genres")
    if isinstance(genres, str) and genres.strip():
        parts.append(f"Generos: {genres.strip()}")

    topics = row.get("topics")
    if isinstance(topics, str) and topics.strip() and topics.strip().lower() != "nan":
        parts.append(f"Temas: {topics.strip()}")

    directors = row.get("directors")
    if isinstance(directors, str) and directors.strip():
        parts.append(f"Directores: {directors.strip()}")

    actors = row.get("actors")
    if isinstance(actors, str) and actors.strip():
        actor_list = [a.strip() for a in actors.split("|") if a.strip()][:5]
        if actor_list:
            parts.append(f"Reparto: {', '.join(actor_list)}")

    plot = row.get("plot_clean")
    if isinstance(plot, str) and plot.strip():
        parts.append(plot.strip())

    return ". ".join(parts) if parts else "(no metadata)"

texts = [build_text_input(movies_df.iloc[i]) for i in range(len(movies_df))]
print(f"Built {len(texts):,} text inputs")
print(f"Sample (first movie, truncated): {texts[0][:200]}...")
print(f"Text length stats: mean={np.mean([len(t) for t in texts]):.0f} chars, "
      f"median={np.median([len(t) for t in texts]):.0f} chars")

# ── Bước 2: nạp mô hình và mã hóa văn bản trên GPU ────────────────────────────────────
print(f"\nLoading {TEXT_MODEL_NAME} trên {DEVICE_STR.upper()}...")
st_model = SentenceTransformer(TEXT_MODEL_NAME, device=DEVICE_STR)

if DEVICE.type == "cuda":
    # Giải phóng bộ nhớ đệm trước khi mã hóa để tối ưu VRAM
    torch.cuda.empty_cache()
    print(f"[GPU] VRAM trước encode: "
          f"{torch.cuda.memory_allocated(0)/1e6:.0f} MB allocated / "
          f"{torch.cuda.memory_reserved(0)/1e6:.0f} MB reserved")

text_embeddings = st_model.encode(
    texts,
    batch_size=TEXT_BATCH_SIZE,       # 512 trên GPU, 64 trên CPU
    show_progress_bar=True,
    normalize_embeddings=True,        # L2-norm → cosine sim = dot product
    convert_to_numpy=True,
    device=DEVICE_STR,
)

if DEVICE.type == "cuda":
    print(f"[GPU] VRAM sau encode : "
          f"{torch.cuda.memory_allocated(0)/1e6:.0f} MB allocated")

print(f"\nText embeddings shape : {text_embeddings.shape}")
print(f"Mean L2 norm          : {np.linalg.norm(text_embeddings, axis=1).mean():.4f}")

del st_model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print(f"[GPU] VRAM sau del model: {torch.cuda.memory_allocated(0)/1e6:.0f} MB")


Built 78,628 text inputs
Sample (first movie, truncated): Chaos Theory. Generos: Drama|Comedia. Directores: Marcos Siega. Reparto: Ryan Reynolds, Emily Mortimer, Stuart Townsend, Sarah Chalke, Mike Erwin. Frank (Ryan Reynolds), famoso autor de un best seller...
Text length stats: mean=519 chars, median=487 chars

Loading paraphrase-multilingual-MiniLM-L12-v2 trên CUDA...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[GPU] VRAM trước encode: 471 MB allocated / 484 MB reserved


Batches:   0%|          | 0/154 [00:00<?, ?it/s]

[GPU] VRAM sau encode : 480 MB allocated

Text embeddings shape : (78628, 384)
Mean L2 norm          : 1.0000
[GPU] VRAM sau del model: 480 MB


### 2.11 Gom cụm embedding văn bản
Embedding văn bản được phân cụm bằng KMeans. Với mỗi item, các cụm gần nhất được gán trọng số thông qua softmax trên cosine similarity, cho phép LightFM tiếp nhận tín hiệu ngữ nghĩa ở dạng đặc trưng rời rạc.


In [ ]:
# [Giải thích cell]
# - Mục đích: gom cụm embedding văn bản để biến vector dense thành feature rời rạc cho LightFM.
# - Đầu vào chính: `text_embeddings` đã chuẩn hóa.
# - Kết quả tạo ra: `text_cluster_assignments`, gồm các cụm gần nhất và trọng số của từng phim.
# - Lưu ý: softmax theo temperature giúp phân phối trọng số mượt hơn giữa các cụm gần nhau.

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize as sk_normalize

text_clusters = None
text_cluster_assignments = None  # {movie_idx: [(cluster_id, weight), ...]}

if USE_TEXT_CLUSTERS and text_embeddings is not None:
    print(f"Running KMeans (K={K_TEXT_CLUSTERS}) on text embeddings...")
    kmeans_text = KMeans(
        n_clusters=K_TEXT_CLUSTERS,
        random_state=42,
        n_init=10,
        verbose=0,
    )
    kmeans_text.fit(text_embeddings)

    # L2-normalize centroids để cosine sim = dot product
    centroids_text = sk_normalize(kmeans_text.cluster_centers_, norm="l2", axis=1)

    # Cosine similarity giữa từng movie embedding và mọi centroid
    # text_embeddings đã L2-normalized, centroids cũng vậy → dot product = cosine
    sims_text = text_embeddings @ centroids_text.T  # (n_movies, K_TEXT_CLUSTERS)

    # Lấy top-K cluster cho từng movie
    top_idx = np.argpartition(-sims_text, TEXT_TOP_K, axis=1)[:, :TEXT_TOP_K]

    # Sắp xếp các cụm gần nhất theo từng item
    text_cluster_assignments = []
    for i in range(len(movies_df)):
        clusters = top_idx[i]
        cluster_sims = sims_text[i, clusters]
        order = np.argsort(-cluster_sims)
        clusters_sorted = clusters[order]
        sims_sorted = cluster_sims[order]

        # Tính trọng số softmax có điều chỉnh temperature
        exp_sims = np.exp(sims_sorted / TEXT_TEMP)
        weights = exp_sims / exp_sims.sum()

        text_cluster_assignments.append([
            (int(c), float(w)) for c, w in zip(clusters_sorted, weights)
        ])

    # Kiểm tra chẩn đoán
    cluster_sizes = np.bincount(top_idx[:, 0], minlength=K_TEXT_CLUSTERS)
    print(f"  Top-1 cluster size: min={cluster_sizes.min()}, "
          f"max={cluster_sizes.max()}, mean={cluster_sizes.mean():.1f}")
    print(f"  Sample movie 0 top-{TEXT_TOP_K} text clusters: {text_cluster_assignments[0]}")
    print(f"  Sample movie 100 top-{TEXT_TOP_K} text clusters: {text_cluster_assignments[100]}")
else:
    print("Text clustering: SKIPPED (USE_TEXT_CLUSTERS=False)")

Running KMeans (K=200) on text embeddings...
  Top-1 cluster size: min=80, max=840, mean=393.1
  Sample movie 0 top-3 text clusters: [(88, 0.37377652525901794), (30, 0.3185960054397583), (24, 0.30762746930122375)]
  Sample movie 100 top-3 text clusters: [(93, 0.3551696240901947), (103, 0.3312114179134369), (180, 0.31361904740333557)]


### 2.12 Thành phần KGE bị vô hiệu hóa
Toàn bộ pipeline Knowledge Graph Embedding bị loại bỏ trong thí nghiệm này. Các biến liên quan đến KG, TransE, item KGE và user KGE được đặt về `None` để bảo đảm không có tín hiệu KGE đi vào LightFM.


In [ ]:
# [Giải thích cell]
# - Mục đích: vô hiệu hóa nhánh Knowledge Graph Embedding trong biến thể không KGE.
# - Kết quả tạo ra: các artifact KG/KGE được đặt bằng `None` để không đi vào LightFM.
# - Lưu ý: cell này là điểm kiểm soát ablation, không phải lỗi thiếu dữ liệu.

# [ABLATION] KGE hoàn toàn bị loại bỏ — đặt các biến liên quan = None
kg_cluster_assignments       = None  # item KGE clusters — disabled
user_kg_cluster_assignments  = None  # user KGE clusters — disabled

print("[Ablation-NoKGE] Mục 9 (KGE pipeline) SKIPPED.")
print("  kg_cluster_assignments      = None")
print("  user_kg_cluster_assignments = None")


### 2.13 Tích hợp đặc trưng Tier 2 không KGE
Chỉ các cụm text được tích hợp vào `item_feat_dicts`. Cụm item KGE và user KGE bị bỏ qua theo cờ ablation, giúp cô lập đóng góp của embedding văn bản.


In [ ]:
# [Giải thích cell]
# - Mục đích: đưa các feature Tier 2 vào từ điển đặc trưng item/user trước khi fit LightFM.
# - Đầu vào chính: text clusters, KGE clusters và các cờ ablation tương ứng.
# - Kết quả tạo ra: `item_feat_dicts` và/hoặc `user_feat_dicts` đã có thêm feature embedding dạng cụm.
# - Lưu ý: trọng số feature được nhân scale để cân bằng ảnh hưởng giữa các nhóm đặc trưng.

# ══════════════════════════════════════════════════════════════════════════════
# Inject ITEM Tier 2: chỉ Text cluster features
# ══════════════════════════════════════════════════════════════════════════════

# 1. Text cluster features → item_feat_dicts
n_text_added = 0
if text_cluster_assignments is not None:
    for i in range(len(movies_df)):
        mid = int(movies_df.iloc[i]["id"])
        for cluster_id, weight in text_cluster_assignments[i]:
            item_feat_dicts[mid][f"text_cluster:{cluster_id}"] = weight
    n_text_added = K_TEXT_CLUSTERS

# 2. KGE cluster features — DISABLED (ablation)
n_kg_added = 0  # [ABLATION] không inject kg_cluster

# 3. User KGE cluster features — DISABLED (ablation)
n_user_kg_added = 0  # [ABLATION] không inject user_kg_cluster

# ══════════════════════════════════════════════════════════════════════════════
# Tóm tắt kiểm tra
# ══════════════════════════════════════════════════════════════════════════════
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

n_text_tags = len({t for t in all_item_tags if t.startswith('text_cluster:')})
n_kg_tags   = len({t for t in all_item_tags if t.startswith('kg_cluster:')})
n_user_kg_tags = len({t for t in all_user_tags if t.startswith('user_kg_cluster:')})
n_tier3_tags   = len(all_user_tags) - n_user_kg_tags

print("=" * 65)
print("Feature Space Tóm tắt kiểm tra — [Ablation: No KGE]")
print("=" * 65)
print(f"ITEM features: {len(all_item_tags)} total")
print(f"  ├─ Tier 1 (categorical + continuous) : {len(all_tier1_tags)}")
print(f"  ├─ Tier 2 text clusters              : {n_text_tags}")
print(f"  └─ Tier 2 item KGE clusters          : {n_kg_tags}  [ABLATION: disabled]")
print()
print(f"USER features: {len(all_user_tags)} total")
print(f"  ├─ Tier 3 (Twitter behavioral)       : {n_tier3_tags}")
print(f"  └─ Tier 2 user KGE clusters          : {n_user_kg_tags}  [ABLATION: disabled]")


### 2.14 Lưu checkpoint sau tiền xử lý
Checkpoint lưu lại dữ liệu đã tiền xử lý, đặc trưng item/user và text clusters. Việc lưu checkpoint giúp tái lập nhanh thí nghiệm không KGE.


In [ ]:
# [Giải thích cell]
# - Mục đích: lưu các artifact sau tiền xử lý để tái sử dụng ở các lần chạy sau.
# - Đầu vào chính: dataframes, từ điển đặc trưng, cấu hình và artifact embedding nếu có.
# - Kết quả tạo ra: một file `.pkl` trong thư mục checkpoint.
# - Lưu ý: checkpoint giúp tránh chạy lại các bước tốn thời gian như encoder, KMeans hoặc TransE.

ABLATION_CHECKPOINT = os.path.join(CHECKPOINT_DIR, "ablation_no_kge_preprocessed.pkl")

artifacts = {
    "ratings_df":    ratings_df,
    "users_df":      users_df,
    "movies_df":     movies_df,
    "train_df":      train_df,
    "valid_df":      valid_df,
    "test_df":       test_df,
    "test_warm_df":  test_warm_df,
    "test_cold_df":  test_cold_df,
    # Đặc trưng item
    "item_feat_dicts":          item_feat_dicts,
    "all_item_tags":            all_item_tags,
    "all_tier1_tags":           all_tier1_tags,
    "text_cluster_assignments": text_cluster_assignments,
    # Đặc trưng user (KGE clusters absent)
    "user_feat_dicts":               user_feat_dicts,
    "all_user_tags":                  all_user_tags,
    # Cấu hình
    "config": {
        "POSITIVE_THRESHOLD": POSITIVE_THRESHOLD, "MIN_RATINGS": MIN_RATINGS,
        "NO_COMPONENTS": NO_COMPONENTS, "LEARNING_RATE": LEARNING_RATE,
        "ITEM_ALPHA": ITEM_ALPHA, "USER_ALPHA": USER_ALPHA,
        "MAX_EPOCHS": MAX_EPOCHS, "PATIENCE": PATIENCE,
        "NUM_THREADS": NUM_THREADS, "K_VALUES": K_VALUES,
        "K_TEXT_CLUSTERS": K_TEXT_CLUSTERS, "TEXT_TOP_K": TEXT_TOP_K, "TEXT_TEMP": TEXT_TEMP,
        "USE_TEXT_CLUSTERS": USE_TEXT_CLUSTERS,
        "USE_KG_CLUSTERS": USE_KG_CLUSTERS,
        "USE_USER_KG_CLUSTERS": USE_USER_KG_CLUSTERS,
    },
}

with open(ABLATION_CHECKPOINT, "wb") as f:
    pickle.dump(artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(ABLATION_CHECKPOINT) / (1024 * 1024)
print(f"[Checkpoint] Saved: {ABLATION_CHECKPOINT}  ({size_mb:.1f} MB)")
print(f"[Checkpoint] Artifacts: {list(artifacts.keys())}")


### 2.15 Nạp checkpoint
Cell này nạp lại checkpoint của biến thể không KGE và khôi phục các biến cần thiết cho LightFM. Các artifact KGE không được nạp vì không tồn tại trong thiết lập ablation.


In [ ]:
# [Giải thích cell]
# - Mục đích: nạp lại checkpoint đã lưu để tiếp tục từ bước xây dựng ma trận LightFM.
# - Đầu vào chính: file `.pkl` chứa dữ liệu và đặc trưng đã tiền xử lý.
# - Kết quả tạo ra: các biến notebook được khôi phục về trạng thái sau preprocessing.
# - Lưu ý: cần bảo đảm checkpoint khớp với đúng biến thể thực nghiệm đang chạy.

import os, pickle, ast, warnings
from collections import Counter
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

LOAD_FROM_CHECKPOINT = True

if LOAD_FROM_CHECKPOINT:
    checkpoint_name = "ablation_no_kge_preprocessed.pkl"
    checkpoint_candidates = [
        os.path.join(".", checkpoint_name),
        checkpoint_name,
        os.path.join("/content", checkpoint_name),
        os.path.join("/kaggle/working/checkpoints", checkpoint_name),
    ]
    checkpoint_path = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Không tìm thấy checkpoint '{checkpoint_name}'. "
            "Vui lòng chạy các Mục 0–10 trước hoặc upload file checkpoint."
        )

    with open(checkpoint_path, "rb") as f:
        artifacts = pickle.load(f)

    ratings_df   = artifacts["ratings_df"]
    users_df     = artifacts["users_df"]
    movies_df    = artifacts["movies_df"]
    train_df     = artifacts["train_df"]
    valid_df     = artifacts["valid_df"]
    test_df      = artifacts["test_df"]
    test_warm_df = artifacts["test_warm_df"]
    test_cold_df = artifacts["test_cold_df"]

    item_feat_dicts          = artifacts["item_feat_dicts"]
    all_item_tags            = artifacts["all_item_tags"]
    all_tier1_tags           = artifacts["all_tier1_tags"]
    text_cluster_assignments = artifacts.get("text_cluster_assignments")

    user_feat_dicts = artifacts["user_feat_dicts"]
    all_user_tags   = artifacts["all_user_tags"]

    # KGE artifacts absent in this ablation
    kg_cluster_assignments      = None
    user_kg_cluster_assignments = None

    cfg = artifacts.get("config", {})
    POSITIVE_THRESHOLD = cfg.get("POSITIVE_THRESHOLD", 7)
    MIN_RATINGS        = cfg.get("MIN_RATINGS", 10)
    NO_COMPONENTS      = cfg.get("NO_COMPONENTS", 64)
    LEARNING_RATE      = cfg.get("LEARNING_RATE", 0.05)
    ITEM_ALPHA         = cfg.get("ITEM_ALPHA", 1e-6)
    USER_ALPHA         = cfg.get("USER_ALPHA", 1e-6)
    MAX_EPOCHS         = cfg.get("MAX_EPOCHS", 30)
    PATIENCE           = cfg.get("PATIENCE", 5)
    NUM_THREADS        = cfg.get("NUM_THREADS", 4)
    K_VALUES           = tuple(cfg.get("K_VALUES", (5, 10, 20, 50)))
    K_TEXT_CLUSTERS    = cfg.get("K_TEXT_CLUSTERS", 200)
    TEXT_TOP_K         = cfg.get("TEXT_TOP_K", 3)
    TEXT_TEMP          = cfg.get("TEXT_TEMP", 0.1)

    print(f"[Load] Checkpoint: {checkpoint_path}")
    print(f"[Load] Train/Valid/Test: {len(train_df):,} / {len(valid_df):,} / {len(test_df):,}")
    print(f"[Load] Item feature tags: {len(all_item_tags):,}")
    print(f"[Load] User feature tags: {len(all_user_tags):,}")
    print(f"[Load] KGE clusters     : DISABLED (ablation)")
    print(f"[Load] Cấu hình: components={NO_COMPONENTS}, lr={LEARNING_RATE}, epochs={MAX_EPOCHS}")
else:
    print("LOAD_FROM_CHECKPOINT=False → dùng dữ liệu từ các Mục trên.")


[Load] Checkpoint: ./ablation_no_kge_preprocessed.pkl
[Load] Train/Valid/Test: 962,280 / 474 / 474
[Load] Item feature tags: 790
[Load] User feature tags: 19
[Load] KGE clusters     : DISABLED (ablation)
[Load] Config: components=64, lr=0.05, epochs=30


### 2.15 Khởi tạo `Dataset` của LightFM
Toàn bộ tên đặc trưng item và user được gom lại để fit đối tượng `Dataset`. Ma trận đặc trưng được xây dựng với `normalize=False` vì trọng số giữa các nhóm đặc trưng đã được hiệu chỉnh trước đó.


In [ ]:
# [Giải thích cell]
# - Mục đích: cài LightFM, thư viện được dùng để huấn luyện mô hình gợi ý lai.
# - Lưu ý: cell này không huấn luyện mô hình; chỉ bảo đảm package có sẵn trong runtime.

pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831125 sha256=c923746a5562fbf4c886d9eb01832a2614f388672bc6a69621e27a0e451975a8
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [ ]:
# [Giải thích cell]
# - Mục đích: chuyển từ điển đặc trưng sang định dạng sparse matrix mà LightFM yêu cầu.
# - Đầu vào chính: danh sách user_id, movie_id, tên feature và các feature dict.
# - Kết quả tạo ra: `dataset`, `user_feature_matrix` và `item_feature_matrix`.
# - Lưu ý: `normalize=False` giữ nguyên trọng số đã hiệu chỉnh ở các bước trước.

from lightfm import LightFM
from lightfm.data import Dataset

# Refresh tag sets từ dicts hiện tại
all_item_tags = set()
for feats in item_feat_dicts.values():
    all_item_tags.update(feats.keys())

all_user_tags = set()
for feats in user_feat_dicts.values():
    all_user_tags.update(feats.keys())

n_user_kg_tags = 0  # [ABLATION] user KGE clusters disabled
n_tier3_tags   = len(all_user_tags)

# Fit đối tượng Dataset
dataset = Dataset()
dataset.fit(
    users=users_df["id"].tolist(),
    items=movies_df["id"].tolist(),
    user_features=list(all_user_tags),
    item_features=list(all_item_tags),
)

# Build feature matrices (normalize=False: weights đã được calibrated per Tier)
item_features_raw = [(mid, feats) for mid, feats in item_feat_dicts.items()]
user_features_raw = [(uid, feats) for uid, feats in user_feat_dicts.items()]

item_feature_matrix = dataset.build_item_features(item_features_raw, normalize=False)
user_feature_matrix = dataset.build_user_features(user_features_raw, normalize=False)

print(f"User feature matrix : {user_feature_matrix.shape}")
print(f"  ├─ Tier 3 (Twitter behavioral)  : {n_tier3_tags} features")
print(f"  └─ Tier 2 user KGE clusters      : {n_user_kg_tags} features  ← MỚI")
print(f"Item feature matrix : {item_feature_matrix.shape}")
print(f"  ├─ Tier 1 (categorical+cont)     : {len(all_tier1_tags)} features")
n_text = len({t for t in all_item_tags if t.startswith('text_cluster:')})
n_kg   = len({t for t in all_item_tags if t.startswith('kg_cluster:')})
print(f"  ├─ Tier 2 text clusters          : {n_text} features")
print(f"  └─ Tier 2 item KGE clusters      : {n_kg} features")
print()
print(f"User feature density: {user_feature_matrix.nnz / (user_feature_matrix.shape[0]*user_feature_matrix.shape[1])*100:.4f}%")
print(f"Item feature density: {item_feature_matrix.nnz / (item_feature_matrix.shape[0]*item_feature_matrix.shape[1])*100:.4f}%")


User feature matrix : (474, 493)
  ├─ Tier 3 (Twitter behavioral)  : 19 features
  └─ Tier 2 user KGE clusters      : 0 features  ← MỚI
Item feature matrix : (78628, 79418)
  ├─ Tier 1 (categorical+cont)     : 590 features
  ├─ Tier 2 text clusters          : 200 features
  └─ Tier 2 item KGE clusters      : 0 features

User feature density: 1.1071%
Item feature density: 0.0133%


### 2.16 Xây dựng interactions nhị phân
Tương tác dương được định nghĩa bởi $r_{u,i} \geq \tau$. Các tương tác dưới ngưỡng không được đưa vào ma trận huấn luyện, giữ thống nhất với thiết lập phản hồi ẩn của baseline.


In [ ]:
# [Giải thích cell]
# - Mục đích: chuyển rating thành ma trận tương tác nhị phân cho LightFM.
# - Đầu vào chính: một split dữ liệu và ngưỡng `POSITIVE_THRESHOLD`.
# - Kết quả tạo ra: sparse interaction matrix chỉ chứa các rating dương.
# - Lưu ý: các rating dưới ngưỡng bị bỏ qua, phù hợp với thiết lập phản hồi ẩn.

def build_interactions_binary(df, dataset, threshold=POSITIVE_THRESHOLD):
    positives = df[df["rate"] >= threshold]
    if len(positives) == 0:
        return None
    pairs = [(int(r["id"]), int(r["movie_id"])) for _, r in positives.iterrows()]
    interactions, _ = dataset.build_interactions(pairs)
    return interactions

train_interactions = build_interactions_binary(train_df, dataset)
valid_inter        = build_interactions_binary(valid_df, dataset)
test_inter         = build_interactions_binary(test_df, dataset)
test_warm_inter    = build_interactions_binary(test_warm_df, dataset)
test_cold_inter    = build_interactions_binary(test_cold_df, dataset)

def nnz(m): return m.nnz if m is not None else 0

print(f"Threshold = {POSITIVE_THRESHOLD} (binary implicit, no sample_weight)")
print(f"Train interactions : {nnz(train_interactions):,}")
print(f"Valid interactions : {nnz(valid_inter):,}")
print(f"Test  interactions : {nnz(test_inter):,}")
print(f"  ├─ Warm items    : {nnz(test_warm_inter):,}")
print(f"  └─ Cold items    : {nnz(test_cold_inter):,}")

Threshold = 7 (binary implicit, no sample_weight)
Train interactions : 412,438
Valid interactions : 240
Test  interactions : 262
  ├─ Warm items    : 253
  └─ Cold items    : 9


## 3. Thực nghiệm và Kết quả (Experiments & Results)

### 3.1 Các chỉ số đánh giá Top-$K$
Các chỉ số Precision@$K$, Recall@$K$, NDCG@$K$, HR@$K$ và MRR@$K$ được tính sau khi mask item đã xuất hiện trong train. Cùng một bộ chỉ số được sử dụng cho baseline, ablation và mô hình đề xuất.


In [ ]:
# [Giải thích cell]
# - Mục đích: định nghĩa hàm đánh giá Top-K dùng chung cho validation và test.
# - Đầu vào chính: ground truth held-out, lịch sử train và hàm/mô hình sinh điểm gợi ý.
# - Kết quả tạo ra: Precision@K, Recall@K, NDCG@K, HR@K và MRR@K.
# - Lưu ý: item đã xuất hiện trong train được mask để tránh gợi ý lại lịch sử cũ.

def evaluate_metrics(model, test_interactions, train_interactions,
                     user_features, item_features,
                     k_values=(5, 10, 20), num_threads=1):
    if test_interactions is None or test_interactions.nnz == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0., "hr": 0., "mrr": 0.}
                for k in k_values}

    test_csr  = test_interactions.tocsr()
    train_csr = train_interactions.tocsr()
    n_users, n_items = test_csr.shape
    max_k = max(k_values)

    accum = {k: {"precision": [], "recall": [], "ndcg": [], "hr": [], "mrr": []}
             for k in k_values}

    for u in range(n_users):
        true_items = set(test_csr[u].indices.tolist())
        if not true_items:
            continue

        scores = model.predict(
            user_ids=u, item_ids=np.arange(n_items),
            user_features=user_features, item_features=item_features,
            num_threads=num_threads,
        )

        train_items = train_csr[u].indices
        scores[train_items] = -np.inf

        top_idx = np.argpartition(-scores, max_k)[:max_k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        relevance  = np.array([1.0 if i in true_items else 0.0 for i in top_idx])
        n_relevant = len(true_items)

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(k):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    return {
        k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}
        for k, metrics in accum.items()
    }


def print_metrics(metrics, label):
    print(f"\n=== {label} ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k in sorted(metrics.keys()):
        m = metrics[k]
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")

### 3.2 Huấn luyện LightFM với WARP
LightFM được huấn luyện theo từng epoch bằng WARP loss, với early stopping dựa trên NDCG@10 của tập validation. Bảng 1 là lịch sử huấn luyện được tạo bởi cell bên dưới.


In [ ]:
# [Giải thích cell]
# - Mục đích: huấn luyện LightFM theo từng epoch và theo dõi hiệu năng validation.
# - Đầu vào chính: ma trận tương tác train, feature user/item và siêu tham số WARP.
# - Kết quả tạo ra: `model`, `best_model`, `best_epoch` và lịch sử huấn luyện.
# - Lưu ý: early stopping dựa trên NDCG@10 giúp giảm nguy cơ quá khớp.

model = LightFM(
    loss="warp",
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=42,
)

best_ndcg10      = -1.0
patience_counter = 0
history          = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.fit_partial(
        interactions=train_interactions,
        user_features=user_feature_matrix,
        item_features=item_feature_matrix,
        epochs=1,
        num_threads=1,
        verbose=False,
    )

    valid_metrics = evaluate_metrics(
        model, valid_inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )

    ndcg10 = valid_metrics[10]["ndcg"]
    history.append({
        "epoch": epoch,
        **{f"ndcg@{k}": valid_metrics[k]["ndcg"] for k in K_VALUES}
    })
    print(f"Epoch {epoch:>2} | "
          + " | ".join(f"NDCG@{k}: {valid_metrics[k]['ndcg']:.4f}" for k in K_VALUES))

    if ndcg10 > best_ndcg10:
        best_ndcg10      = ndcg10
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\n[PROPOSED-FINAL] Best Valid NDCG@10: {best_ndcg10:.4f}")

Epoch  1 | NDCG@5: 0.0060 | NDCG@10: 0.0073 | NDCG@20: 0.0116 | NDCG@50: 0.0183
Epoch  2 | NDCG@5: 0.0105 | NDCG@10: 0.0105 | NDCG@20: 0.0169 | NDCG@50: 0.0224
Epoch  3 | NDCG@5: 0.0130 | NDCG@10: 0.0144 | NDCG@20: 0.0155 | NDCG@50: 0.0250
Epoch  4 | NDCG@5: 0.0143 | NDCG@10: 0.0169 | NDCG@20: 0.0190 | NDCG@50: 0.0277
Epoch  5 | NDCG@5: 0.0167 | NDCG@10: 0.0209 | NDCG@20: 0.0229 | NDCG@50: 0.0305
Epoch  6 | NDCG@5: 0.0144 | NDCG@10: 0.0156 | NDCG@20: 0.0198 | NDCG@50: 0.0299
Epoch  7 | NDCG@5: 0.0122 | NDCG@10: 0.0148 | NDCG@20: 0.0212 | NDCG@50: 0.0271
Epoch  8 | NDCG@5: 0.0159 | NDCG@10: 0.0159 | NDCG@20: 0.0241 | NDCG@50: 0.0314
Epoch  9 | NDCG@5: 0.0178 | NDCG@10: 0.0190 | NDCG@20: 0.0241 | NDCG@50: 0.0299
Epoch 10 | NDCG@5: 0.0157 | NDCG@10: 0.0206 | NDCG@20: 0.0269 | NDCG@50: 0.0334

Early stopping at epoch 10

[PROPOSED-FINAL] Best Valid NDCG@10: 0.0209


In [ ]:
# [Giải thích cell]
# - Mục đích: hiển thị lịch sử huấn luyện dưới dạng bảng để quan sát quá trình hội tụ.
# - Đầu vào chính: danh sách `history` được ghi lại sau mỗi epoch.
# - Kết quả tạo ra: `history_df`, bảng thể hiện loss/metric validation theo epoch.

history_df = pd.DataFrame(history)
display(history_df)

,epoch,ndcg@5,ndcg@10,ndcg@20,ndcg@50
0,1,0.006035,0.007350,0.011636,0.018308
1,2,0.010491,0.010491,0.016872,0.022410
2,3,0.013046,0.014434,0.015454,0.025011
3,4,0.014294,0.016863,0.018999,0.027725
4,5,0.016667,0.020854,0.022944,0.030516
5,6,0.014369,0.015623,0.019841,0.029917
6,7,0.012211,0.014804,0.021191,0.027089
7,8,0.015906,0.015906,0.024134,0.031447
8,9,0.017758,0.018962,0.024083,0.029912
9,10,0.015674,0.020602,0.026903,0.033373


### 3.3 Đánh giá cuối trên validation và test
Kết quả được báo cáo trên validation, test-full, test-warm và test-cold. Bảng 2 hỗ trợ so sánh hiệu năng tổng thể và khả năng xử lý item cold-start.


In [ ]:
# [Giải thích cell]
# - Mục đích: chạy đánh giá cuối cùng trên các split validation/test đã xác định.
# - Đầu vào chính: mô hình đã huấn luyện, train interactions và các tập held-out.
# - Kết quả tạo ra: bảng metric cho full, warm và cold split nếu có.
# - Lưu ý: không dùng kết quả test để chọn siêu tham số.

n_user_kg_tags = 0  # [ABLATION]
n_tier3_tags   = len(all_user_tags)

print("=" * 70)
print("FINAL EVALUATION — Ablation: No KGE")
print(f"Cấu hình : threshold={POSITIVE_THRESHOLD}, components={NO_COMPONENTS}")
print(f"Item   : {len(all_item_tags)} features (Tier1 + TextCluster [No KGE])")
print(f"User   : {len(all_user_tags)} features (Tier3={n_tier3_tags} [No User KGE])")
print("=" * 70)

for label, inter in [
    ("VALID",     valid_inter),
    ("TEST",      test_inter),
    ("TEST_WARM", test_warm_inter),
    ("TEST_COLD", test_cold_inter),
]:
    metrics = evaluate_metrics(
        model, inter, train_interactions,
        user_feature_matrix, item_feature_matrix,
        k_values=K_VALUES, num_threads=NUM_THREADS,
    )
    print_metrics(metrics, label)


FINAL EVALUATION — Ablation: No KGE
Config : threshold=7, components=64
Item   : 790 features (Tier1 + TextCluster [No KGE])
User   : 19 features (Tier3=19 [No User KGE])

=== VALID ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0042 |   0.0208 |   0.0157 |   0.0208 |   0.0139
  10 |   0.0038 |   0.0375 |   0.0206 |   0.0375 |   0.0157
  20 |   0.0031 |   0.0625 |   0.0269 |   0.0625 |   0.0174
  50 |   0.0019 |   0.0958 |   0.0334 |   0.0958 |   0.0184

=== TEST ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0038 |   0.0191 |   0.0144 |   0.0191 |   0.0127
  10 |   0.0023 |   0.0229 |   0.0157 |   0.0229 |   0.0134
  20 |   0.0019 |   0.0382 |   0.0196 |   0.0382 |   0.0145
  50 |   0.0013 |   0.0649 |   0.0251 |   0.0649 |   0.0154

=== TEST_WARM ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
---

### 3.4 Suy luận cho một người dùng
Cell này minh họa quy trình sinh top-$N$ gợi ý cho một người dùng cụ thể. Các item đã xuất hiện trong lịch sử train được loại bỏ khỏi danh sách ứng viên.


In [ ]:
# [Giải thích cell]
# - Mục đích: minh họa quy trình suy luận top-N cho một người dùng cụ thể.
# - Đầu vào chính: user_id, mô hình hoặc danh sách popularity, metadata phim và train history.
# - Kết quả tạo ra: bảng phim được đề xuất sau khi loại bỏ item người dùng đã thấy.
# - Lưu ý: đây là ví dụ định tính, không thay thế cho đánh giá định lượng Top-K.

def recommend_for_user(user_id, model, dataset, movies_df,
                       user_feature_matrix, item_feature_matrix,
                       train_df, n_recs=10):
    uid_map, _, iid_map, _ = dataset.mapping()
    if user_id not in uid_map:
        print(f"User {user_id} không tồn tại.")
        return pd.DataFrame()

    u_idx   = uid_map[user_id]
    n_items = item_feature_matrix.shape[0]

    scores = model.predict(
        user_ids=u_idx, item_ids=np.arange(n_items),
        user_features=user_feature_matrix, item_features=item_feature_matrix,
    )

    seen = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    seen_indices = [iid_map[m] for m in seen if m in iid_map]
    scores[seen_indices] = -np.inf

    top_indices   = np.argsort(-scores)[:n_recs]
    idx2movie     = {v: k for k, v in iid_map.items()}
    top_movie_ids = [idx2movie[i] for i in top_indices]

    result = movies_df[movies_df["id"].isin(top_movie_ids)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["predicted_score"] = [scores[iid_map[mid]] for mid in result["id"]]
    return result.sort_values("predicted_score", ascending=False)


sample_user = train_df["id"].iloc[0]
print(f"[PROPOSED-FINAL] Gợi ý top-10 cho user_id = {sample_user}\n")
recs = recommend_for_user(
    user_id=sample_user, model=model, dataset=dataset,
    movies_df=movies_df,
    user_feature_matrix=user_feature_matrix,
    item_feature_matrix=item_feature_matrix,
    train_df=train_df, n_recs=10,
)
print(recs.to_string(index=False))

[PROPOSED-FINAL] Gợi ý top-10 cho user_id = 103007

    id             original_title                                          genres  year_published  rate  predicted_score
721028            Rosemary's Baby                                    Terror|Drama          1968.0   7.7      -266.908691
800220                    Aladdin Animación|Fantástico|Musical|Infantil|Aventuras          1992.0   7.4      -267.091156
491709                    Amadeus                                           Drama          1984.0   7.7      -267.190308
173696             Shutter Island                                Thriller|Intriga          2010.0   7.6      -267.199554
315125                Funny Games                                        Thriller          1997.0   7.4      -267.419586
431773   Looney Tunes (TV Series)          Serie de TV|Animación|Comedia|Infantil          1930.0   7.1      -267.505646
218925       Mr. Bean (TV Series)                             Serie de TV|Comedia          1990.0   7

## 4. Kết luận (Conclusion)
Notebook này hoàn thiện quy trình thực nghiệm cho biến thể không KGE. Các bước tiền xử lý, chia dữ liệu, huấn luyện và đánh giá được giữ thống nhất với baseline để kết quả có thể so sánh trực tiếp. Khi đặt cạnh các notebook baseline và ablation còn lại, kết quả của notebook này hỗ trợ phân tích đóng góp tương đối của metadata, embedding văn bản và embedding đồ thị tri thức đối với chất lượng gợi ý Top-$K$.
